# FIR Custom Embedder — Fixed Training Loop

**Theme**: Retrieval Augmented Generation over Kannada FIR Dataset

**What this notebook does**:
1. Loads Udupi crime FIR data
2. Preprocesses Kannada text
3. Trains a custom SimpleEmbedderV4 with attention pooling
4. Fixed training loop — 3 pairs per crime type per epoch
5. Builds FAISS index
6. Hybrid BM25 + Embedding search
7. RAG generation with Qwen2

## Cell 1 — Install Dependencies

In [ ]:
!pip install faiss-cpu rank_bm25 sentence-transformers --quiet

## Cell 2 — Mount Google Drive

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/MAHE/MSIS Coursework/OddSem2025MAHE/Share/Agentic Workshop/NoStudentAccess/Data/'
else:
    DATA_DIR = '../Data/'

print(f'DATA_DIR set to: {DATA_DIR}')

## Cell 3 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import math
import random
import re
from tqdm import tqdm
import unicodedata
from collections import defaultdict, Counter

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

# use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Cell 4 — Load Data

In [ ]:
file = DATA_DIR + 'UdupiCrimeData.csv'
df = pd.read_csv(file, header=0).dropna()
df.reset_index(inplace=True)

print('Udupi Crime Dataset')
print('-' * 30)
print(f'Number of records  : {df.shape[0]}')
print(f'Number of features : {df.shape[1]}')
print(f'Crime types        : {df["Crime Type"].nunique()}')
print()
df.head(3)

## Cell 5 — Kannada Text Preprocessing

Strips everything outside Kannada Unicode range U+0C80–U+0CFF.
Removes dates, IPC numbers, English text, punctuation.

In [ ]:
def normalize_text(text):
    text = str(text)
    # keep only Kannada unicode range + whitespace
    text = re.sub(r'[^\u0C80-\u0CFF\s]', '', text)
    # collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def tokenize(text):
    return normalize_text(text).split()

# test it
sample = df['Crime Description'].iloc[0]
print('Original:')
print(sample[:200])
print()
print('Normalized:')
print(normalize_text(sample)[:200])
print()
print('Tokens:')
print(tokenize(sample)[:10])

## Cell 6 — Build Texts List

In [ ]:
# normalize all crime descriptions
texts = df['Crime Description'].apply(normalize_text).tolist()
print(f'Total texts: {len(texts)}')
print(f'Sample: {texts[0][:100]}')

## Cell 7 — Build Vocabulary + Crime Groups

- `vocab_counter` → all unique Kannada tokens (used to build word2idx)
- `crime_groups`  → tokens grouped by crime type (used for contrastive training pairs)

In [ ]:
crime_groups  = defaultdict(list)
vocab_counter = Counter()

for idx, row in df.iterrows():
    tokens = tokenize(texts[idx])
    vocab_counter.update(tokens)
    crime_groups[row['Crime Type']].append(tokens)

print(f'Vocabulary size  : {len(vocab_counter)}')
print(f'Crime types      : {len(crime_groups)}')
print()
print('Crime type distribution:')
for ct, docs in sorted(crime_groups.items(), key=lambda x: -len(x[1])):
    print(f'  {ct[:30]:30s} → {len(docs)} FIRs')

## Cell 8 — Build word2idx / idx2word

In [ ]:
word2idx = {w: i for i, w in enumerate(vocab_counter.keys())}
idx2word = {i: w for w, i in word2idx.items()}

print(f'word2idx size: {len(word2idx)}')
# show first 5 entries
for w, i in list(word2idx.items())[:5]:
    print(f'  "{w}" → {i}')

## Cell 9 — encode_doc Function

Converts a list of tokens into a list of integer indices using word2idx.
Unknown tokens are skipped safely.

In [ ]:
def encode_doc(tokens):
    """Convert token list to integer index list.
    Accepts either a list of tokens or a raw string.
    """
    # fix: handle raw string input by tokenizing first
    if isinstance(tokens, str):
        tokens = tokenize(tokens)
    return [word2idx[t] for t in tokens if t in word2idx]

# test
sample_tokens = tokenize(texts[0])
encoded = encode_doc(sample_tokens)
print(f'Token count  : {len(sample_tokens)}')
print(f'Encoded count: {len(encoded)}')
print(f'First 5 indices: {encoded[:5]}')

## Cell 10 — SimpleEmbedderV4 (Attention Pooling + Deep Network)

Architecture:
```
Input indices
    ↓ nn.Embedding (vocab_size × 64)     — Layer 1: word lookup
    ↓ nn.Linear(64 → 1) + Softmax        — Layer 2: attention scorer
    ↓ weighted sum                        — learned pooling
    ↓ nn.Linear(64 → 128) + ReLU         — Layer 3: expand
    ↓ Dropout(0.1)
    ↓ nn.Linear(128 → 64)                — Layer 4: compress
Output: 64-dim document vector
```

In [ ]:
class SimpleEmbedderV4(nn.Module):
    def __init__(self, vocab_size, dim=64):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, dim)
        # attention: learns importance score per word
        self.attention = nn.Linear(dim, 1)
        # feedforward network
        self.network   = nn.Sequential(
            nn.Linear(dim, 128),
            nn.ReLU(),
            nn.Dropout(0.1),       # reduced from 0.2 for small dataset
            nn.Linear(128, dim)
        )

    def forward(self, x):
        # Step 1: lookup word vectors
        embedded = self.embed(x)                      # (n_tokens, 64)

        # Step 2: compute importance score per token
        scores  = self.attention(embedded)             # (n_tokens, 1)
        weights = torch.softmax(scores, dim=0)         # (n_tokens, 1)

        # Step 3: weighted sum instead of plain mean
        pooled  = (weights * embedded).sum(dim=0)      # (64,)

        # Step 4: transform through network
        return self.network(pooled)                    # (64,)

# quick shape test
embed_dim = 64
test_model = SimpleEmbedderV4(len(word2idx), dim=embed_dim)
test_input = torch.tensor(encode_doc(tokenize(texts[0])))
test_out   = test_model(test_input)
print(f'Input shape : {test_input.shape}')
print(f'Output shape: {test_out.shape}')   # should be torch.Size([64])
del test_model, test_input, test_out

## Cell 11 — Initialize Model, Optimizer, Loss

In [ ]:
embed_dim = 64

model     = SimpleEmbedderV4(len(word2idx), dim=embed_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.CosineEmbeddingLoss()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters : {total_params:,}')
print(f'Embedding table  : {len(word2idx) * embed_dim:,} params')
print(f'Attention layer  : {embed_dim * 1:,} params')
print(f'Network layers   : {embed_dim*128 + 128*embed_dim:,} params')

## Cell 12 — Fixed Training Loop

**Key fixes over original**:
1. `encode_doc` + forward + backward all inside the `for _ in range(3)` loop
2. Empty encoding guard — skips docs where all tokens are OOV
3. 500 epochs — loss was still dropping at 100
4. 3 pairs per crime type per epoch — 3x more gradient updates, smoother curve

In [ ]:
num_epochs = 500
epoch_loss = []

for epoch in range(num_epochs):
    total_loss = 0.0

    for crime_type, docs in crime_groups.items():
        # need at least 2 docs to form a positive pair
        if len(docs) < 2:
            continue

        # --- 3 pairs per crime type per epoch ---
        for _ in range(3):

            # positive pair — same crime type
            d1_tokens, d2_tokens = random.sample(docs, 2)

            # negative pair — different crime type
            neg_crime  = random.choice([
                ct for ct in crime_groups.keys()
                if ct != crime_type
            ])
            d3_tokens  = random.choice(crime_groups[neg_crime])

            # encode tokens → integer tensors
            d1 = torch.tensor(encode_doc(d1_tokens)).to(device)
            d2 = torch.tensor(encode_doc(d2_tokens)).to(device)
            d3 = torch.tensor(encode_doc(d3_tokens)).to(device)

            # skip if any doc encoded to empty (all tokens OOV)
            if len(d1) == 0 or len(d2) == 0 or len(d3) == 0:
                continue

            # forward pass — mean pooled + network
            e1 = model(d1)
            e2 = model(d2)
            e3 = model(d3)

            # contrastive losses
            pos_label = torch.tensor(1.0).to(device)
            neg_label = torch.tensor(-1.0).to(device)
            loss_pos  = loss_fn(e1, e2, pos_label)   # push together
            loss_neg  = loss_fn(e1, e3, neg_label)   # push apart
            loss      = loss_pos + loss_neg

            # backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

    epoch_loss.append(total_loss)

    # print every 10 epochs
    if epoch % 10 == 0 or epoch == num_epochs - 1:
        print(f'Epoch {epoch:4d} | Loss: {total_loss:.4f}')

print('Training complete!')

## Cell 13 — Plot Loss Curve

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
fig.tight_layout(pad=4.0)

ax.plot(epoch_loss, 'b', linewidth=1.5, label='Training Loss')

# moving average to show trend clearly
window = 20
moving_avg = np.convolve(
    epoch_loss,
    np.ones(window)/window,
    mode='valid'
)
ax.plot(
    range(window-1, len(epoch_loss)),
    moving_avg,
    'r',
    linewidth=2,
    label=f'{window}-epoch moving avg'
)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss Value', fontsize=12)
ax.set_title('Loss vs. Epoch — SimpleEmbedderV4', fontsize=14)
ax.legend()
ax.grid(alpha=0.2)
plt.show()

print(f'Final loss  : {epoch_loss[-1]:.4f}')
print(f'Best loss   : {min(epoch_loss):.4f} at epoch {epoch_loss.index(min(epoch_loss))}')

## Cell 14 — Generate Custom Embeddings Matrix

In [ ]:
model.eval()   # switch off dropout for inference

X_customembeddings = np.empty((df.shape[0], embed_dim))

with torch.no_grad():   # no gradient tracking needed at inference
    for i in tqdm(range(df.shape[0]), desc='Generating embeddings'):
        tokens  = tokenize(texts[i])          # fix: tokenize first
        encoded = encode_doc(tokens)

        if len(encoded) == 0:
            # fallback: zero vector if no valid tokens
            X_customembeddings[i] = np.zeros(embed_dim)
            continue

        tensor = torch.tensor(encoded).to(device)
        X_customembeddings[i] = model(tensor).cpu().numpy()

print(f'Embeddings matrix shape: {X_customembeddings.shape}')

## Cell 15 — Build FAISS Index (Custom Embeddings)

In [ ]:
import faiss

dim          = X_customembeddings.shape[1]   # 64
custom_index = faiss.IndexFlatL2(dim)
custom_index.add(X_customembeddings.astype('float32'))

print(f'FAISS index dimension  : {dim}')
print(f'Vectors indexed        : {custom_index.ntotal}')

## Cell 16 — Semantic Search (Custom Embeddings)

In [ ]:
def semantic_search(query, k=5):
    model.eval()
    tokens  = tokenize(query)              # fix: tokenize first
    encoded = encode_doc(tokens)

    if len(encoded) == 0:
        print('No valid tokens in query after preprocessing')
        return pd.Series([])

    with torch.no_grad():
        q_emb = model(
            torch.tensor(encoded).to(device)
        ).cpu().numpy().reshape(1, -1).astype('float32')

    distances, indices = custom_index.search(q_emb, k)

    results = df.iloc[indices[0]]['Crime Description']
    return results, distances[0]

# test
query   = 'ಮೊಬೈಲ್ ಫೋನ್ ಕಳವು'
results, dists = semantic_search(query, k=3)
print(f'Query: {query}\n')
for i, (desc, dist) in enumerate(zip(results, dists)):
    print(f'Result {i+1} | Distance: {dist:.4f}')
    print(desc[:200])
    print('─' * 50)

## Cell 17 — Build BM25 Index

In [ ]:
from rank_bm25 import BM25Okapi

# tokenize entire corpus for BM25
tokenized_corpus = [tokenize(t) for t in tqdm(texts, desc='Tokenizing for BM25')]
bm25             = BM25Okapi(tokenized_corpus)

print(f'BM25 index built over {len(tokenized_corpus)} documents')

## Cell 18 — Hybrid Search (BM25 + Custom Embeddings)

- `alpha = 1.0` → pure BM25 (keyword)
- `alpha = 0.0` → pure embedding (semantic)
- `alpha = 0.5` → equal mix (recommended starting point)

In [ ]:
def hybrid_search(query, k=5, alpha=0.5):
    query_tokens = tokenize(query)

    # ── BM25 scores ──────────────────────────────────────────
    bm25_scores  = bm25.get_scores(query_tokens)

    # ── Embedding scores ─────────────────────────────────────
    encoded = encode_doc(query_tokens)
    if len(encoded) == 0:
        # fallback to pure BM25 if no valid tokens
        top_k = bm25_scores.argsort()[-k:][::-1]
        return df.iloc[top_k]['Crime Description'], bm25_scores[top_k]

    model.eval()
    with torch.no_grad():
        q_emb = model(
            torch.tensor(encoded).to(device)
        ).cpu().numpy().reshape(1, -1).astype('float32')

    # search all documents
    distances, indices = custom_index.search(q_emb, len(texts))

    # convert L2 distance → similarity (smaller distance = more similar)
    emb_scores              = np.zeros(len(texts))
    emb_scores[indices[0]]  = 1 / (1 + distances[0])

    # ── Normalize both to 0-1 ────────────────────────────────
    def normalize(scores):
        min_s = scores.min()
        max_s = scores.max()
        if max_s - min_s < 1e-9:
            return scores
        return (scores - min_s) / (max_s - min_s)

    bm25_norm = normalize(bm25_scores)
    emb_norm  = normalize(emb_scores)

    # ── Combine ───────────────────────────────────────────────
    combined  = alpha * bm25_norm + (1 - alpha) * emb_norm
    top_k     = combined.argsort()[-k:][::-1]

    return df.iloc[top_k]['Crime Description'], combined[top_k]

# test
query   = 'ಮೊಬೈಲ್ ಫೋನ್ ಕಳವು'
results, scores = hybrid_search(query, k=3, alpha=0.5)
print(f'Query: {query}\n')
for i, (desc, score) in enumerate(zip(results, scores)):
    print(f'Result {i+1} | Score: {score:.4f}')
    print(desc[:200])
    print('─' * 50)

## Cell 19 — Compare All Search Methods

In [ ]:
def compare_search_methods(query, k=3):
    print(f'Query: {query}')
    print('=' * 70)

    methods = [
        ('BM25 Only',       1.0),
        ('Embedding Only',  0.0),
        ('Hybrid (50/50)',  0.5),
    ]

    for method_name, alpha in methods:
        print(f'\n── {method_name} (alpha={alpha}) ──')
        results, scores = hybrid_search(query, k=k, alpha=alpha)
        for i, (desc, score) in enumerate(zip(results, scores)):
            print(f'  [{i+1}] score={score:.3f} | {desc[:150]}...')

    print()

# run on multiple queries
test_queries = [
    'ಮೊಬೈಲ್ ಫೋನ್ ಕಳವು',
    'ಮನೆಯಲ್ಲಿ ಕಳ್ಳತನ',
    'ರಸ್ತೆ ಅಪಘಾತ',
    'ಗಾಂಜಾ ಸೇವನೆ',
]

for q in test_queries:
    compare_search_methods(q, k=2)
    print()

## Cell 20 — Tune Alpha Parameter

In [ ]:
alphas = [0.0, 0.2, 0.3, 0.5, 0.7, 0.8, 1.0]
query  = 'ಮೊಬೈಲ್ ಫೋನ್ ಕಳವು'

print(f'Alpha tuning for query: {query}\n')
print(f'{"Alpha":>6} | {"Top Score":>10} | Top Result (first 120 chars)')
print('-' * 70)

for alpha in alphas:
    results, scores = hybrid_search(query, k=1, alpha=alpha)
    top_desc = list(results)[0][:120]
    print(f'{alpha:>6.1f} | {scores[0]:>10.4f} | {top_desc}...')

## Cell 21 — Save Model

In [ ]:
save_path = '/content/drive/MyDrive/fir_embedder_v4.pth'

torch.save({
    'model_state_dict' : model.state_dict(),
    'word2idx'         : word2idx,
    'idx2word'         : idx2word,
    'embed_dim'        : embed_dim,
    'vocab_size'       : len(word2idx),
    'epoch_loss'       : epoch_loss,
}, save_path)

print(f'Model saved to {save_path}')

## Cell 22 — Load Model (run this to resume without retraining)

In [ ]:
load_path  = '/content/drive/MyDrive/fir_embedder_v4.pth'
checkpoint = torch.load(load_path, map_location=device)

# restore vocab
word2idx   = checkpoint['word2idx']
idx2word   = checkpoint['idx2word']
embed_dim  = checkpoint['embed_dim']

# rebuild model and load weights
model      = SimpleEmbedderV4(checkpoint['vocab_size'], dim=embed_dim).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# restore loss history
epoch_loss = checkpoint['epoch_loss']

print(f'Model loaded from {load_path}')
print(f'Vocab size : {len(word2idx)}')
print(f'Embed dim  : {embed_dim}')
print(f'Trained for: {len(epoch_loss)} epochs')
print(f'Final loss : {epoch_loss[-1]:.4f}')